In [4]:
rst = """51~京东方Ａ~000725~5.81~5.86~5.93~12589567~5021579~7567988~5.80~64087~5.79~36683~5.78~42420~5.77~17658~5.76~20559~5.81~17905~5.82~80434~5.83~79912~5.84~58066~5.85~60483~~20260814161427~-0.05~-0.85~5.94~5.71~5.81/12589567/7316940134~12589567~731694~3.56~36.17~~5.94~5.71~3.92~2054.83~2152.28~1.61~6.45~5.27~0.80~-115393~5.81~31.52~36.75~~~1.32~731694.0134~628.4096~10816~ A~GP-A~39.87~-4.28~0.96~4.37~1.10~9.50~3.73~5.44~-4.28~25.38~35367134117~37044328064~-24.13~49.97~35367134117~~~51.14~0.17~~CNY~0~~5.71~73348~"""

In [ ]:
# http://qt.gtimg.cn/q=sz000725
rst1


In [1]:
import pandas as pd
import numpy as np
import random
import unicodedata
from datetime import datetime, timedelta
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ===================== 1. 基础配置与数据生成 =====================
# 设置随机种子，保证结果可复现
np.random.seed(42)
random.seed(42)

# 总记录数
RECORD_COUNT = 50

# 生成基础数据
# 1. 整数类型
id_list = list(range(1, RECORD_COUNT + 1))
age_list = np.random.randint(18, 61, size=RECORD_COUNT)
order_count_list = np.random.randint(0, 101, size=RECORD_COUNT)

# 2. 浮点数类型
monthly_income_list = np.random.uniform(3000, 50000, size=RECORD_COUNT).round(2)
avg_score_list = np.random.uniform(0, 5, size=RECORD_COUNT).round(1)
discount_rate_list = np.random.uniform(0, 1, size=RECORD_COUNT).round(4)
account_balance_list = np.random.uniform(0, 100000, size=RECORD_COUNT).round(2)

# 3. 字符串类型
# 随机中文姓名生成
surnames = ['赵', '钱', '孙', '李', '周', '吴', '郑', '王', '冯', '陈', '褚', '卫', '蒋', '沈', '韩', '杨', '朱', '秦', '尤', '许']
given_names = ['伟', '芳', '娜', '秀英', '敏', '静', '丽', '强', '磊', '军', '洋', '艳', '勇', '娟', '杰', '娟', '涛', '明', '超', '秀英']
name_list = [random.choice(surnames) + random.choice(given_names) + random.choice(given_names) if random.random() > 0.3 else random.choice(surnames) + random.choice(given_names) for _ in range(RECORD_COUNT)]

# 国内城市列表
cities = ['北京', '上海', '广州', '深圳', '杭州', '南京', '成都', '重庆', '武汉', '西安', '郑州', '济南', '青岛', '沈阳', '大连']
city_list = [random.choice(cities) for _ in range(RECORD_COUNT)]

# 手机号生成
phone_list = ['1' + random.choice(['3', '5', '7', '8', '9']) + ''.join(random.choices('0123456789', k=9)) for _ in range(RECORD_COUNT)]

# 邮箱生成
email_list = [f'user{id}@example.com' for id in id_list]

# 标签生成
tag_list = [random.choice(['购物', '旅行', '美食', '运动', '数码', '美妆', '母婴', '汽车']) + ',' + random.choice(['新人', '活跃', '高价值', '沉睡']) for _ in range(RECORD_COUNT)]

# 备注长文本
remark_list = [f'这是第{id}条测试记录的备注信息，用于测试长文本格式的显示效果，包含中文、数字和符号等内容。' for id in id_list]

# 4. 日期时间类型
# 注册日期（2020-2025年随机日期）
start_date = datetime(2020, 1, 1)
end_date = datetime(2025, 12, 31)
date_range = end_date - start_date
register_date_list = [start_date + timedelta(days=random.randint(0, date_range.days)) for _ in range(RECORD_COUNT)]

# 创建时间（带时分秒）
create_time_list = [d + timedelta(hours=random.randint(0, 23), minutes=random.randint(0, 59), seconds=random.randint(0, 59)) for d in register_date_list]

# 5. 布尔类型
is_vip_list = np.random.choice([True, False], size=RECORD_COUNT, p=[0.3, 0.7])
is_active_list = np.random.choice([True, False], size=RECORD_COUNT, p=[0.8, 0.2])

# 6. 分类类型
member_levels = ['普通会员', '黄金会员', '铂金会员', '钻石会员']
member_level_list = np.random.choice(member_levels, size=RECORD_COUNT, p=[0.4, 0.3, 0.2, 0.1])

# 7. 空值测试列（30%概率空值）
null_test_list = [np.nan if random.random() < 0.3 else f'测试内容{id}' for id in id_list]

# ===================== 2. 创建DataFrame =====================
df = pd.DataFrame({
    'ID': id_list,
    '姓名': name_list,
    '年龄': age_list,
    '所在城市': city_list,
    '手机号': phone_list,
    '邮箱': email_list,
    '是否会员': is_vip_list,
    '会员等级': member_level_list,
    '月收入': monthly_income_list,
    '账户余额': account_balance_list,
    '平均评分': avg_score_list,
    '折扣率': discount_rate_list,
    '订单数': order_count_list,
    '注册日期': register_date_list,
    '创建时间': create_time_list,
    '是否激活': is_active_list,
    '标签': tag_list,
    '备注': remark_list,
    '空值测试列': null_test_list
})

df.to_pickle("df.pkl")


In [ ]:
df['邮箱'].str.split('@',expand=True)[0]

In [ ]:
df['手机号'].str.split('@',expand=True)[0]

In [ ]:
import a_trade_calendar
# import pymysql
from datetime import datetime,date,timedelta
from utils.dbManager import MySQLHelper
import pandas as pd
import numpy as np
import tushare as ts
from IPython.display import clear_output
ts.set_token('34485b6ded61906921d22121a51f9e197ac95640b9b6003dedb8cc3e')
def lookUpColName(df):
    db = MySQLHelper(db='jhd')
    cols = db.qry2df(f"""select distinct cols.COLUMN_NAME,SUBSTRING_INDEX(cols.COLUMN_COMMENT,'~',1) as COLUMN_COMMENT \
                    from information_schema.`COLUMNS` as cols \
                    where cols.TABLE_SCHEMA = 'jhd'""")
    cols['COLUMN_COMMENT'] = cols.COLUMN_COMMENT.str.decode('utf-8')
    colmap = cols['COLUMN_COMMENT']
    colmap.index = cols['COLUMN_NAME']
    return df.columns.map(colmap)

In [ ]:
#初始化交易日历 ，直接运行，就可以把补充插入还没有的日历数据
import a_trade_calendar
# import pymysql
from datetime import datetime,date,timedelta
from utils.dbManager import MySQLHelper
import pandas as pd
# import numpy as np

def get_all_trading_dates_optimized(start_date_str='2026-01-01'): #获取交易日期
    """
    优化版本：利用库的 get_next_trade_date 功能快速迭代。
    这个方法比逐日检查效率更高。

    Args:
        start_date_str (str): 开始日期字符串，格式 'YYYY-MM-DD'。插入的日期是这个日期的下一个交易日期

    Returns:
        list: 交易日期字符串列表，格式 'YYYY-MM-DD'。
    """
    current_date = pd.to_datetime(start_date_str)
    current_date_str = current_date.strftime("%Y-%m-%d")
    tody = pd.to_datetime(date.today() )
    tody_str = tody.strftime("%Y-%m-%d")

    trading_dates = []
    # current_date_str = start_date
    db = MySQLHelper(db='jhd')
    df = db.qry2df('select max(trade_date) as trade_date from jhd.trading_dates')
    if pd.isna(df.iloc[0,0]):
        #如果原表中的日历为空，那么就使用指定的 start_date_str
        # 如果起始日不是交易日，则找到它之后的第一个交易日作为起点
        current_date_str = a_trade_calendar.get_next_trade_date(current_date_str,cnt=1)
        current_date = pd.to_datetime(current_date_str)
    else:
        current_date_str = df.iloc[0,0].strftime("%Y-%m-%d")
        current_date_str = a_trade_calendar.get_next_trade_date(current_date_str,cnt=1)
        current_date = pd.to_datetime(current_date_str)
    
    # 循环获取后续交易日，直到超过今天
    while current_date_str <= tody_str:
        trading_dates.append(current_date)
        try:
            # 获取下一个交易日
            current_date_str = a_trade_calendar.get_next_trade_date(current_date_str, 1)
            current_date = pd.to_datetime(current_date_str)
        except Exception as e:
            # 如果出现日期耗尽的提示，说明已达到内置日历的末尾
            if "更新" in str(e) or "upgrade" in str(e).lower():
                print(f"日历数据已到达尽头或需要更新: {e}")
                break
            else:
                raise e  # 如果是其他错误，则重新抛出
    df = pd.DataFrame({"trade_date":trading_dates})
    print(f"""最大日期是：{current_date_str}""")
    if len(df) >= 1:
        df['trade_date'] = df['trade_date'].dt.date
    return df

def instTradingDates(): #插入交易日期到mysql
    df = get_all_trading_dates_optimized()
    if len(df) >= 1:
        db = MySQLHelper(db='jhd')
        db.bulkInsert('trading_dates',df=df)
        print(f"共插入 {len(df)} 个交易日历。")
    else:
        print(f"没有获取到交易日历数据")

def insert_to_mysql(trading_dates, db_config): # 没有使用
    """
    将交易日期列表插入到MySQL数据库。

    Args:
        trading_dates (list): 交易日期列表。
        db_config (dict): 包含数据库连接信息的字典。
    """
    connection = None
    try:
        # 建立数据库连接
        connection = pymysql.connect(
            host=db_config['host'],
            port=db_config['port'],
            user=db_config['user'],
            password=db_config['password'],
            database=db_config['database'],
            charset='utf8mb4'
        )

        cursor = connection.cursor()

        # 创建表的SQL语句 (如果表不存在)
        create_table_sql = """
        CREATE TABLE IF NOT EXISTS trading_dates (
            id INT AUTO_INCREMENT PRIMARY KEY,
            trade_date DATE NOT NULL UNIQUE,
            INDEX idx_trade_date (trade_date)
        ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci;
        """
        cursor.execute(create_table_sql)
        print("检查/创建表 'trading_dates' 完成。")

        # 准备插入SQL语句 (使用 INSERT IGNORE 以防万一有重复数据)
        insert_sql = "INSERT IGNORE INTO trading_dates (trade_date) VALUES (%s)"

        # 将字符串日期转换为 datetime.date 对象以便 MySQL 正确接收
        date_objects = [datetime.strptime(d, '%Y-%m-%d').date() for d in trading_dates]

        # 批量插入数据
        cursor.executemany(insert_sql, [(d,) for d in date_objects])

        connection.commit()
        print(f"成功插入 {cursor.rowcount} 条交易日期记录到数据库。")

    except Exception as e:
        print(f"数据库操作发生错误: {e}")
        if connection:
            connection.rollback()
    finally:
        if connection:
            cursor.close()
            connection.close()
            print("数据库连接已关闭。")


if __name__ == "__main__":
    # 1. 配置数据库连接信息，请根据您的实际情况修改
    # db_config = {
    #     'host': 'localhost',  # MySQL服务器地址
    #     'port': 3306,  # MySQL端口
    #     'user': 'root',  # 您的用户名
    #     'password': 'H123456',  # 您的密码
    #     'database': 'stock'  # 您的数据库名
    # }

    print("正在使用 a-trade-calendar 库获取A股历史交易日期...")
    instTradingDates()
    # df = get_all_trading_dates_optimized(start_date_str='2005-01-01')
    # print(f"共获取到 {len(df)} 个交易日。")

    # db = MySQLHelper(db='jhd')
    # db.bulk_insert('trading_dates',df=df)
    # # print("\n正在将交易日期插入到MySQL数据库...")
    # # insert_to_mysql(trading_dates, db_config)

    print("\n程序执行完毕。")

正在使用 a-trade-calendar 库获取A股历史交易日期...
最大日期是：2026-08-11
✅ 批量插入成功，共11条
共插入 11 个交易日历。

程序执行完毕。


In [ ]:
ts.set_token('34485b6ded61906921d22121a51f9e197ac95640b9b6003dedb8cc3e')
pro = ts.pro_api()
# df3 = pro.dividend(end_date='20260412',fields=f'''ts_code, end_date, ann_date, div_proc,stk_div,stk_bo_rate,stk_co_rate, cash_div,cash_div_tax
#             ,record_date, ex_date, pay_date, div_listdate,imp_ann_date, base_date, base_share''')
df = pro.trade_cal(exchange='', start_date='20260501', end_date='20260506')

In [3]:
# 获取并更新 股票列表清单数据：直接运行
import tushare as ts
ts.set_token('34485b6ded61906921d22121a51f9e197ac95640b9b6003dedb8cc3e')
pro = ts.pro_api()
#查询当前所有正常上市交易的股票列表
df = pro.stock_basic(fields=f"""ts_code,symbol,name,area,industry,fullname,enname,cnspell,market,exchange,curr_type,list_status,
    list_date,delist_date,is_hs,act_name,act_ent_type""") 
# exchange='', list_status='L' , fields='ts_code,symbol,name,area,industry,list_date')
#实际导出的列有：ts_code,symbol,name,area,industry,cnspell,market,list_date,act_name,act_ent_type
df['list_date'] = pd.to_datetime(df["list_date"], format="%Y%m%d").dt.date
df['delist_date'] = pd.to_datetime(df["delist_date"], format="%Y%m%d").dt.date

df = df.where(pd.notna(df), None)
df = df.replace({np.nan: None})  # 替换所有 NaN 为 None
#列名称：['ts_code', 'symbol', 'name', 'area', 'industry', 'fullname', 'enname','cnspell', 'market', 
#'exchange', 'curr_type', 'list_status','list_date', 'delist_date', 'is_hs', 'act_name', 'act_ent_type']

db = MySQLHelper(db='jhd')
db.bulkInsUpd(table='ts_stock_basic',df=df)

✅ insert_or_update 执行完成 | 影响行数：77 | 总数据：5539


In [ ]:
#比较两个DF是否完全一样
df2 = db.qry2df(f"""select ts_code,symbol,name,area,industry,fullname,enname,cnspell,market,\
    exchange,curr_type,list_status,list_date,delist_date,is_hs,act_name,act_ent_type from ts_stock_basic """)
df = df.sort_values(by='ts_code')
df2 = df2[df2.ts_code.isin(df.ts_code)]
df2 = df2.sort_values(by='ts_code')
df2.index = df.index
cond = (df == df2) | (df.isna() & df2.isna())
df[~cond.all(axis=1)]

In [16]:
#获取历史每天交易数据：根据上面的交易日历，输入最小的股票交易日期
import tushare as ts
def get_daily_deal(start_date_str='2026-01-01'):
    """
    优化版本：利用库的 get_next_trade_date 功能快速迭代。
    这个方法比逐日检查效率更高。
    Args:
        start_date_str (str): 开始日期字符串，格式 'YYYY-MM-DD'。插入的日期是这个日期的下一个交易日期
    Returns:
        list: 交易日期字符串列表，格式 'YYYY-MM-DD'。
    """
    #先获得还没有 交易
    # current_date = pd.to_datetime(start_date_str)
    # current_date_str = current_date.strftime("%Y-%m-%d")
    # tody = pd.to_datetime(date.today() )
    # tody_str = tody.strftime("%Y-%m-%d")
    db = MySQLHelper(db='jhd')
    df = db.qry2df(f"""select td.id,td.trade_date,td.imp_day_deal 
        from jhd.trading_dates td left join jhd.ts_stock_daily_his tsd on td.trade_date = tsd.trade_date
        where td.trade_date >= '{start_date_str}'
        and tsd.trade_date is NULL """)

    ts.set_token('34485b6ded61906921d22121a51f9e197ac95640b9b6003dedb8cc3e')
    pro = ts.pro_api()

    # df = pro.daily(ts_code='001283.SZ', start_date='20260320', end_date='20260323')
    #或者：
    # df = pro.query('daily', ts_code='000001.SZ', start_date='20180701', end_date='20180718')
    #多个股票
    # df = pro.daily(ts_code='001283.SZ,600000.SH', start_date='20180701', end_date='20180718')
    
    #也可以通过日期取历史某一天的全部历史
    # df = pro.daily(trade_date='20260324')
    df['trade_date'] = pd.to_datetime(df.trade_date)
    df['trade_date'] = df["trade_date"].dt.strftime("%Y%m%d")
    for i in range(len(df)):
        print(f"获取日期：{df.iloc[i,1]}的数据")
        df2 = pro.daily(trade_date=df.iloc[i,1])
        df2['trade_date'] = pd.to_datetime(df2['trade_date'],format='%Y%m%d').dt.date
        df2.columns = ['ts_code', 'trade_date', 'open_price', 'high', 'low', 'close_price',
               'pre_close', 'change_amount', 'pct_chg', 'vol', 'amount']
        # mp = {'ts_code':'ts_code','trade_date':'trade_date','open':'open_price','high':'high',
        #  'low':'low','close':'close_price','pre_close':'pre_close','change':'change_amount',
        # 'pct_chg':'pct_chg','vol':'vol','amount':'amount'}
        # df2.columns.map(mp)
        if len(df2) >= 1:
            db.bulkInsert(table='ts_stock_daily_his',df=df2)
            return len(df2)
    
    print('完成数据插入')
    return 0

if __name__ == "__main__":
    # 1. 配置数据库连接信息，请根据您的实际情况修改
    # db_config = {
    #     'host': 'localhost',  # MySQL服务器地址
    #     'port': 3306,  # MySQL端口
    #     'user': 'root',  # 您的用户名
    #     'password': 'H123456',  # 您的密码
    #     'database': 'stock'  # 您的数据库名
    # }

    print("正在获取A股历史交易记录...")
    get_daily_deal(start_date_str='2026-01-01')
    # df = get_all_trading_dates_optimized(start_date_str='2005-01-01')
    # print(f"共获取到 {len(df)} 个交易日。")

    # db = MySQLHelper(db='jhd')
    # db.bulk_insert('trading_dates',df=df)
    # # print("\n正在将交易日期插入到MySQL数据库...")
    # # insert_to_mysql(trading_dates, db_config)

    print("\n程序执行完毕。")


正在获取A股历史交易记录...
完成数据插入

程序执行完毕。


In [ ]:
#获取分红记录函数，输入分红中的日期（可能是多种数据中的日期）
#执行获取股票分红函数，每分钟只能获取100次数据 # 2026-04-10
#从2026-05-21起 每分钟只能访问1次了。 所以就没有更新了
import tushare as ts
import time
from utils.dbManager import MySQLHelper
from IPython.display import clear_output
def get_dividend(df=None,m=None,n=None,cols=None,tdate=None): 
    #df是传入进来的 股票代码
    #tdate 所有日期都获取一遍数据，格式：yyyymmdd
    ts.set_token('34485b6ded61906921d22121a51f9e197ac95640b9b6003dedb8cc3e')
    pro = ts.pro_api()
    db = MySQLHelper(db='jhd')
    if tdate is None:
        if len(df.loc[m:n-1,:]) > 100:
            print('长度大于100')
            return 0
        # df = db.qry2df("select ts_code from jhd.ts_stock_basic where ts_code not in (select distinct ts_code from ts_dividend)  LIMIT 100")
        for i in range(m,n): #不含n
            df3 = pro.dividend(ts_code=df.loc[i,cols[0]],fields=f'''ts_code, end_date, ann_date, div_proc,stk_div,stk_bo_rate,stk_co_rate, cash_div,cash_div_tax
                ,record_date, ex_date, pay_date, div_listdate,imp_ann_date, base_date, base_share''')
            df3 = df3.where(pd.notna(df3), None)
            df3 = df3.replace({np.nan: None})  # 替换所有 NaN 为 None
            for col in df3.columns:
                # 专门处理 datetime 类型列   下面的条件：np.issubdtype(df3[col].dtype, np.dtype('datetime64')) or 
                if col in ['end_date','ann_date','record_date','ex_date','pay_date','div_listdate','imp_ann_date','base_date']:
                    df3[col] = pd.to_datetime(df3[col], format="%Y%m%d").dt.date
                    df3[col] = df3[col].astype(object).where(df3[col].notna(), None)
                # 处理其他类型列
                else:
                    df3[col] = df3[col].where(df3[col].notna(), None)
            df3 = df3.drop_duplicates()
            df4 = df3.loc[:,['ts_code']].drop_duplicates()
            db.delFromDf(table='tmp_dividend',df=df4)
            db.bulkInsert(table='tmp_dividend',df=df3)  #如果是初始化，可以先插入到tmp_dividend
            
        df.loc[m:n-1,cols[1]] = pd.Timestamp.now().floor('s')
        return df.loc[m:n-1,cols]
    else: #把所有日期都查找一遍
        df3 = pro.dividend(end_date=tdate,fields=f'''ts_code, end_date, ann_date, div_proc,stk_div,stk_bo_rate,stk_co_rate, cash_div,cash_div_tax
                ,record_date, ex_date, pay_date, div_listdate,imp_ann_date, base_date, base_share''')
        df4 = pro.dividend(ann_date=tdate,fields=f'''ts_code, end_date, ann_date, div_proc,stk_div,stk_bo_rate,stk_co_rate, cash_div,cash_div_tax
                ,record_date, ex_date, pay_date, div_listdate,imp_ann_date, base_date, base_share''')
        if not (df3.empty and df4.empty):
            df3 = pd.concat([x for x in [df3, df4] if not x.empty],axis=0,ignore_index=True)
        
        df4 = pro.dividend(record_date=tdate,fields=f'''ts_code, end_date, ann_date, div_proc,stk_div,stk_bo_rate,stk_co_rate, cash_div,cash_div_tax
                ,record_date, ex_date, pay_date, div_listdate,imp_ann_date, base_date, base_share''')
        if not (df3.empty and df4.empty):
            df3 = pd.concat([x for x in [df3, df4] if not x.empty],axis=0,ignore_index=True)
        df4 = pro.dividend(ex_date=tdate,fields=f'''ts_code, end_date, ann_date, div_proc,stk_div,stk_bo_rate,stk_co_rate, cash_div,cash_div_tax
                ,record_date, ex_date, pay_date, div_listdate,imp_ann_date, base_date, base_share''')
        if not (df3.empty and df4.empty):
            df3 = pd.concat([x for x in [df3, df4] if not x.empty],axis=0,ignore_index=True)
        # df4 = pro.dividend(pay_date=tdate,fields=f'''ts_code, end_date, ann_date, div_proc,stk_div,stk_bo_rate,stk_co_rate, cash_div,cash_div_tax
        #         ,record_date, ex_date, pay_date, div_listdate,imp_ann_date, base_date, base_share''')
        # df3 = pd.concat([df3,df4],axis=0,ignore_index=True)
        # df4 = pro.dividend(div_listdate=tdate,fields=f'''ts_code, end_date, ann_date, div_proc,stk_div,stk_bo_rate,stk_co_rate, cash_div,cash_div_tax
        #         ,record_date, ex_date, pay_date, div_listdate,imp_ann_date, base_date, base_share''')
        # df3 = pd.concat([df3,df4],axis=0,ignore_index=True)
        df4 = pro.dividend(imp_ann_date=tdate,fields=f'''ts_code, end_date, ann_date, div_proc,stk_div,stk_bo_rate,stk_co_rate, cash_div,cash_div_tax
                ,record_date, ex_date, pay_date, div_listdate,imp_ann_date, base_date, base_share''')
        if not (df3.empty and df4.empty):
            df3 = pd.concat([x for x in [df3, df4] if not x.empty],axis=0,ignore_index=True)
        # df4 = pro.dividend(base_date=tdate,fields=f'''ts_code, end_date, ann_date, div_proc,stk_div,stk_bo_rate,stk_co_rate, cash_div,cash_div_tax
        #         ,record_date, ex_date, pay_date, div_listdate,imp_ann_date, base_date, base_share''')
        # df3 = pd.concat([df3,df4],axis=0,ignore_index=True)
        df3 = df3.drop_duplicates()

        df4 = df3.loc[:,['ts_code']]
        # df4['divid_date'] = None
        df4 = df4.drop_duplicates().reset_index(drop=True)
 
        return df4

def exe_get_dividend(divid_date=None):
    ts.set_token('34485b6ded61906921d22121a51f9e197ac95640b9b6003dedb8cc3e')
    pro = ts.pro_api()
    db = MySQLHelper(db='jhd')
    start_time = time.time()
    if divid_date is None: #日期为空，那么就是初始化
        df = db.qry2df("select ts_code,divid_date from jhd.ts_stock_basic where divid_date is null")
    else:
        df = get_dividend(tdate=divid_date) #切片后就不能直接更改df的值了
        df['divid_date'] = None
    used_time = time.time() - start_time
    if used_time < 61:
        wait_seconds = 61 - used_time
        time.sleep(wait_seconds)
    
    start_time = time.time()
    m = len(df)//100 if len(df)%100 == 0 else len(df)//100 + 1
    # print(m)
    for i in range(m):
        clear_output(wait=True)
        n = (i+1)*100 if (i+1)*100 <=len(df) else len(df)
        # print(i*100,n)
        t = get_dividend(df=df,m=i*100,n=n,cols=['ts_code','divid_date']) #切片后就不能直接更改df的值了
        t['divid_date'] = pd.to_datetime(t['divid_date']).dt.date
        
        # db.bulk_update(table='ts_stock_basic',df=df.loc[i*100:n-1,['ts_code','divid_date']],key='ts_code')
        db.bulk_update(table='ts_stock_basic',df=t,key='ts_code') #更新分红的更新时间
        #下面是用于调试的，看插入的数据和上面更新的数据是否一致
        db.bulkInsert(table='tmp_ts_code',df=t)
        
        used_time = time.time() - start_time
        if used_time < 61:
            wait_seconds = 61 - used_time
            time.sleep(wait_seconds)
        start_time = time.time()
    return df

if __name__ == "__main__":
    # df = db.qry2df(f"""select DATE_FORMAT(max(trade_date),'%Y%m%d') as trade_date from ts_stock_daily_his """)
    # if len(df) >= 1:
    #     df = exe_get_dividend(divid_date=df.iloc[0,0])
    df = exe_get_dividend(divid_date='20260521')


#还需要在数据库中处理一下。
# select distinct tpd.ts_code ,ttc.ts_code
# from jhd.tmp_dividend tpd right join tmp_ts_code ttc on tpd.ts_code = ttc.ts_code
# where tpd.ts_code is null
# union ALL
# select distinct tpd.ts_code ,ttc.ts_code
# from jhd.tmp_dividend tpd left join tmp_ts_code ttc on tpd.ts_code = ttc.ts_code
# where ttc.ts_code is null
# ---处理每天的分红数据：
# delete from jhd.ts_dividend td
# where td.ts_code in (select ts_code from tmp_ts_code);

# insert into jhd.ts_dividend (ts_code, end_date, ann_date, div_proc,stk_div,stk_bo_rate,stk_co_rate, cash_div,cash_div_tax
#             ,record_date, ex_date, pay_date, div_listdate,imp_ann_date, base_date, base_share )
# select ts_code, end_date, ann_date, div_proc,stk_div,stk_bo_rate,stk_co_rate, cash_div,cash_div_tax
#             ,record_date, ex_date, pay_date, div_listdate,imp_ann_date, base_date, base_share 
# from tmp_dividend tpd ;
# commit;

In [8]:
# 股票分析：
# 1，先看分红数据和股价 判断是否可以购买
df = db.qry2df(f"""SELECT tsd.`name`,tsd.market,tsd.industry, \
        a.close_price,a.open_price,a.high,td.* \
        FROM ts_dividend td \
        left join jhd.ts_stock_basic tsd on td.ts_code = tsd.ts_code \
        LEFT JOIN jhd.ts_stock_daily_his a ON td.ts_code = a.ts_code \
    		and  a.trade_date = ( SELECT MAX(b.trade_date) FROM ts_stock_daily_his b \
				WHERE b.ts_code = td.ts_code and b.trade_date <= td.pay_date ) \
        where td.pay_date >= '2026-04-01'""")
df.columns = lookUpColName(df)



# 2，再看公司提供的商品和服务，对应市场的竞争力

# df = db.qry2df(f"""select * from ts_stock_daily_his where trade_date = '2026-04-10'""")
# df = db.qry2df(f"""select * from ts_stock_basic""")
# df.columns = lookUpColName(df)

In [ ]:
df

In [ ]:
#Yahoo! Finance 的 API 下载市场数据  pip install yfinance
import yfinance as yf
dat = yf.Ticker("MSFT")
#单一股票代码
dat = yf.Ticker("MSFT")
dat.info
dat.calendar
dat.analyst_price_targets
dat.quarterly_income_stmt
dat.history(period='1mo')
dat.option_chain(dat.options[0]).calls

In [ ]:
#多个股票代码
tickers = yf.Tickers('MSFT AAPL GOOG')
tickers.tickers['MSFT'].info
yf.download(['MSFT', 'AAPL', 'GOOG'], period='1mo')

In [ ]:
#基金：
spy = yf.Ticker('SPY').funds_data
spy.description
spy.top_holdings


In [6]:
#调用tushare 的历史日线数据
import tushare as ts
ts.set_token('34485b6ded61906921d22121a51f9e197ac95640b9b6003dedb8cc3e')
pro = ts.pro_api()
df2 = pro.daily(trade_date='20250505')

# df3 = pro.dividend(ts_code='300979.SZ',fields=f'''ts_code, end_date, ann_date, div_proc,stk_div,stk_bo_rate,stk_co_rate, cash_div,cash_div_tax
#                 ,record_date, ex_date, pay_date, div_listdate,imp_ann_date, base_date, base_share''')
# df = pro.daily(ts_code='001283.SZ', start_date='20260320', end_date='20260323')
#或者：
# df = pro.query('daily', ts_code='000001.SZ', start_date='20180701', end_date='20180718')
#多个股票
# df = pro.daily(ts_code='001283.SZ,600000.SH', start_date='20180701', end_date='20180718')

#也可以通过日期取历史某一天的全部历史
# df2 = pro.daily(trade_date='20260324')

#调用tushare 的api接口
# import tushare as ts
# ts.set_token('34485b6ded61906921d22121a51f9e197ac95640b9b6003dedb8cc3e')
# pro = ts.pro_api()

# df = pro.trade_cal(exchange='', start_date='20180901', end_date='20181001', fields='exchange,cal_date,is_open,pretrade_date', is_open='0')
# df = pro.query('trade_cal', exchange='', start_date='20180901', end_date='20181001', fields='exchange,cal_date,is_open,pretrade_date', is_open='0')

In [8]:
df2

,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount


In [ ]:
['ts_code', 'trade_date', 'open_price', 'high', 'low', 'close_price',
               'pre_close', 'change_amount', 'pct_chg', 'vol', 'amount']

In [4]:
#获取历史的每天交易数据：需要改动sql中4处关于日期的信息
from IPython.display import clear_output
import tushare as ts
import time
from utils.dbManager import MySQLHelper
def get_daily_deal(start_date_str='2026-01-01'):
    """
    优化版本：利用库的 get_next_trade_date 功能快速迭代。
    这个方法比逐日检查效率更高。
    Args:
        start_date_str (str): 开始日期字符串，格式 'YYYY-MM-DD'。插入的日期是这个日期的下一个交易日期
    Returns:
        list: 交易日期字符串列表，格式 'YYYY-MM-DD'。
    """
    #先获得还没有 交易
    # current_date = pd.to_datetime(start_date_str)
    # current_date_str = current_date.strftime("%Y-%m-%d")
    # tody = pd.to_datetime(date.today() )
    # tody_str = tody.strftime("%Y-%m-%d")
    db = MySQLHelper(db='jhd')
    df = db.qry2df(f"""select td.id,td.trade_date,td.imp_day_deal \
        from jhd.trading_dates td left join jhd.ts_stock_daily_his_2011 tsd on td.trade_date = tsd.trade_date \
        where td.trade_date >= '{start_date_str}' and td.trade_date < '2012-01-01' \
        and tsd.trade_date is NULL and td.imp_day_deal is NULL """)

    ts.set_token('34485b6ded61906921d22121a51f9e197ac95640b9b6003dedb8cc3e')
    pro = ts.pro_api()

    # df = pro.daily(ts_code='001283.SZ', start_date='20260320', end_date='20260323')
    #或者：
    # df = pro.query('daily', ts_code='000001.SZ', start_date='20180701', end_date='20180718')
    #多个股票
    # df = pro.daily(ts_code='001283.SZ,600000.SH', start_date='20180701', end_date='20180718')
    
    #也可以通过日期取历史某一天的全部历史
    # df = pro.daily(trade_date='20260324')

    df['trade_date'] = pd.to_datetime(df.trade_date)
    df['trade_date'] = df["trade_date"].dt.strftime("%Y%m%d")
    start_time = time.time()
    for i in range(len(df)):
        
        if i % 50 == 0 and i >= 1 :
            clear_output(wait=True)
            used_time = time.time() - start_time
            if used_time < 61:
                wait_seconds = 61 - used_time
                time.sleep(wait_seconds)
            start_time = time.time()

        print(f"获取日期：{df.iloc[i,1]}的数据")
        df2 = pro.daily(trade_date=df.iloc[i,1])
        df2['trade_date'] = pd.to_datetime(df2['trade_date'],format='%Y%m%d').dt.date
        df2.columns = ['ts_code', 'trade_date', 'open_price', 'high', 'low', 'close_price',
               'pre_close', 'change_amount', 'pct_chg', 'vol', 'amount']
        df2 = df2.fillna(np.nan).replace([np.nan, None], [None, None])
        if len(df2) >= 1:
            db.bulkInsert(table='ts_stock_daily_his_2011',df=df2)
    
    print('完成数据插入')
    return 0
    

if __name__ == "__main__":
    # 1. 配置数据库连接信息，请根据您的实际情况修改
    # db_config = {
    #     'host': 'localhost',  # MySQL服务器地址
    #     'port': 3306,  # MySQL端口
    #     'user': 'root',  # 您的用户名
    #     'password': 'H123456',  # 您的密码
    #     'database': 'stock'  # 您的数据库名
    # }

    print("正在获取A股历史交易记录...")
    get_daily_deal(start_date_str='2011-01-01')
    # df = get_all_trading_dates_optimized(start_date_str='2005-01-01')
    # print(f"共获取到 {len(df)} 个交易日。")

    # db = MySQLHelper(db='jhd')
    # db.bulk_insert('trading_dates',df=df)
    # # print("\n正在将交易日期插入到MySQL数据库...")
    # # insert_to_mysql(trading_dates, db_config)

    print("\n程序执行完毕。")

获取日期：20111101的数据
✅ 批量插入成功，共2209条
获取日期：20111102的数据
✅ 批量插入成功，共2205条
获取日期：20111103的数据
✅ 批量插入成功，共2207条
获取日期：20111104的数据
✅ 批量插入成功，共2204条
获取日期：20111107的数据
✅ 批量插入成功，共2190条
获取日期：20111108的数据
✅ 批量插入成功，共2192条
获取日期：20111109的数据
✅ 批量插入成功，共2190条
获取日期：20111110的数据
✅ 批量插入成功，共2183条
获取日期：20111111的数据
✅ 批量插入成功，共2174条
获取日期：20111114的数据
✅ 批量插入成功，共2176条
获取日期：20111115的数据
✅ 批量插入成功，共2170条
获取日期：20111116的数据
✅ 批量插入成功，共2202条
获取日期：20111117的数据
✅ 批量插入成功，共2202条
获取日期：20111118的数据
✅ 批量插入成功，共2188条
获取日期：20111121的数据
✅ 批量插入成功，共2195条
获取日期：20111122的数据
✅ 批量插入成功，共2203条
获取日期：20111123的数据
✅ 批量插入成功，共2211条
获取日期：20111124的数据
✅ 批量插入成功，共2208条
获取日期：20111125的数据
✅ 批量插入成功，共2212条
获取日期：20111128的数据
✅ 批量插入成功，共2194条
获取日期：20111129的数据
✅ 批量插入成功，共2208条
获取日期：20111130的数据
✅ 批量插入成功，共2200条
获取日期：20111201的数据
✅ 批量插入成功，共2207条
获取日期：20111202的数据
✅ 批量插入成功，共2213条
获取日期：20111205的数据
✅ 批量插入成功，共2209条
获取日期：20111206的数据
✅ 批量插入成功，共2217条
获取日期：20111207的数据
✅ 批量插入成功，共2218条
获取日期：20111208的数据
✅ 批量插入成功，共2218条
获取日期：20111209的数据
✅ 批量插入成功，共2214条
获取日期：20111212的数据
✅ 批量插入成功，共2219条
获取日期：20111

In [13]:
import tushare as ts
ts.set_token('34485b6ded61906921d22121a51f9e197ac95640b9b6003dedb8cc3e')
pro = ts.pro_api()
df2 = pro.daily(trade_date='20220107')

In [ ]:
#主营业务构成
import tushare as ts
ts.set_token('34485b6ded61906921d22121a51f9e197ac95640b9b6003dedb8cc3e')
pro = ts.pro_api()
df = pro.fina_mainbz(ts_code='001283.SZ')

In [6]:
#获取年报信息
import time
import requests
import pdfplumber
import pandas as pd

# =========================
# 获取财报 PDF 地址（加固版）
# =========================
def get_stock_report_url(ts_code: str, report_type="年报"):
    try:
        stock_code = ts_code.split(".")[0]
        url = "http://www.cninfo.com.cn/new/hisAnnouncement/query"

        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Referer": "http://www.cninfo.com.cn/",
            "Accept": "application/json, text/plain, */*",
            "Origin": "http://www.cninfo.com.cn",
            "Host": "www.cninfo.com.cn",
        }

        category = "category_ndbg_szsh" if report_type == "年报" else "category_bndbg_szsh"

        data = {
            "stock": stock_code,
            "tabName": "fulltext",
            "pageSize": 30,
            "pageNum": 1,
            "column": "sse",
            "category": category,
            "seDate": "2020-01-01~2025-12-31",
        }

        resp = requests.post(url, data=data, headers=headers, timeout=15)
        resp.raise_for_status()
        j = resp.json()

        if not j or "announcements" not in j:
            return None, None

        for item in j["announcements"]:
            title = item.get("announcementTitle", "")
            if report_type in title and "摘要" not in title and "说明" not in title:
                pdf_url = f"http://static.cninfo.com.cn/{item['adjunctUrl']}"
                return pdf_url, title

        return None, None

    except Exception as e:
        print(f"获取公告失败: {e}")
        return None, None

# =========================
# 下载 PDF
# =========================
def download_pdf(pdf_url, save_path="report.pdf"):
    try:
        resp = requests.get(pdf_url, stream=True, timeout=20)
        with open(save_path, "wb") as f:
            f.write(resp.content)
        return save_path
    except:
        return None

# =========================
# 从 PDF 提取表格
# =========================
def extract_tables(pdf_path):
    tables = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                table = page.extract_table()
                if table:
                    df = pd.DataFrame(table)
                    tables.append(df)
    except:
        pass
    return tables

# =========================
# 自动识别关键表格
# =========================
def find_income_and_profit(tables):
    income_df = None
    profit_df = None

    for df in tables:
        txt = str(df).replace("\n", "")
        if any(k in txt for k in ["主营业务分产品", "分产品", "营业收入构成", "分行业"]):
            income_df = df
        if any(k in txt for k in ["利润表", "净利润", "营业利润"]):
            profit_df = df
    return income_df, profit_df

# =========================
# 主函数（绝对不报错）
# =========================
def get_finance_report_df(ts_code, report_type="年报"):
    print(f"正在获取 {ts_code} {report_type}...")

    url, title = get_stock_report_url(ts_code, report_type)
    if not url:
        print("未找到财报")
        return None, None

    print(f"找到财报: {title}")
    pdf_path = download_pdf(url)
    if not pdf_path:
        return None, None

    tables = extract_tables(pdf_path)
    income_df, profit_df = find_income_and_profit(tables)

    return income_df, profit_df

# =========================
# 测试（你只需要改股票代码）
# =========================
if __name__ == "__main__":
    stock = "600531.SH"
    income_df, profit_df = get_finance_report_df(stock, "年报")

    print("\n===== 营业收入（分产品）=====")
    print(income_df)

    print("\n===== 利润表 =====")
    print(profit_df)

正在获取 600531.SH 年报...
获取公告失败: 'NoneType' object is not iterable
未找到财报

===== 营业收入（分产品）=====
None

===== 利润表 =====
None


In [5]:
# 使用tushare获取主营产品和收入信息
import tushare as ts
import pandas as pd

# 你的 Tushare Token
pro = ts.pro_api("34485b6ded61906921d22121a51f9e197ac95640b9b6003dedb8cc3e")

# =========================
# 1. 获取 主营业务（分产品收入、毛利）
# =========================
def get_fina_income(ts_code):
    df = pro.fina_mainbz(ts_code=ts_code, start_date="20240101", end_date="20241231")
    return df

# =========================
# 2. 获取 财务指标（利润、PE、ROE等）
# =========================
def get_fina_indicator(ts_code):
    df = pro.fina_indicator(ts_code=ts_code, start_date="20240101", end_date="20241231")
    return df

# =========================
# 3. 测试（你要的 600531.SH）
# =========================
if __name__ == "__main__":
    stock = "600531.SH"

    print("====== 主营业务（分产品）======")
    bz_df = get_fina_income(stock)
    print(bz_df[["item", "bz_item", "income", "profit", "gross_profit_ratio"]])

    print("\n====== 利润 & PE & 财务指标 ======")
    fina_df = get_fina_indicator(stock)
    print(fina_df[["ts_code", "trade_date", "pe", "pe_ttm", "profit_to_gr", "total_profit", "net_profit"]])

====== 主营业务（分产品）======


Exception: 抱歉，您没有接口(fina_mainbz)访问权限，权限的具体详情访问：https://tushare.pro/document/1?doc_id=108。

In [10]:

df = db.qry2df("select * from jhd.ts_stock_basic tsd")
df.columns = lookUpColName(df)

In [11]:
df

,股票代码,股票代码,股票名称,地域,所属行业,股票全称,拼音缩写,市场类型,上市状态,上市日期,是否沪深港通标的,实控人名称,实控人企业性质,更新时间,股票英文全称,交易所代码,交易货币,退市日期,分红调度日期
0,000001.SZ,000001,平安银行,深圳,银行,平安银行股份有限公司,PAYH,主板,L,1991-04-03,S,无实际控制人,其他,2026-04-10 18:20:48,"Ping An Bank Co., Ltd.",SZSE,CNY,None,2026-04-10
1,000002.SZ,000002,万科Ａ,深圳,全国地产,万科企业股份有限公司,WKA,主板,L,1991-01-29,S,无实际控制人,其他,2026-04-10 18:20:48,"China Vanke Co.,Ltd.",SZSE,CNY,None,2026-04-10
2,000004.SZ,000004,*ST国华,深圳,软件服务,深圳国华网安科技股份有限公司,*STGH,主板,L,1990-12-01,N,李映彤,民营企业,2026-04-28 20:29:49,Shenzhen GuoHua Network Security Technology Co...,SZSE,CNY,None,2026-04-28
3,000006.SZ,000006,深振业Ａ,深圳,区域地产,深圳市振业(集团)股份有限公司,SZYA,主板,L,1992-04-27,S,深圳市人民政府国有资产监督管理委员会,地方国有企业,2026-04-10 18:20:48,"Shenzhen Zhenye(Group) Co.,Ltd.",SZSE,CNY,None,2026-04-10
4,000007.SZ,000007,全新好,深圳,其他商业,深圳市全新好股份有限公司,QXH,主板,L,1992-04-13,N,无实际控制人,其他,2026-04-29 20:22:19,"Shenzhen Quanxinhao Co.,Ltd",SZSE,CNY,None,2026-04-29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5524,920978.BJ,920978,开特股份,湖北,汽车配件,湖北开特汽车电子电器系统股份有限公司,KTGF,北交所,L,2023-09-28,N,郑海法,民营企业,2026-04-10 12:13:19,Hubei Kait Automotive Electronic & Electrical ...,BSE,CNY,None,None
5525,920981.BJ,920981,晶赛科技,安徽,元器件,安徽晶赛科技股份有限公司,JSKJ,北交所,L,2021-11-15,N,侯诗益,民营企业,2026-04-10 12:13:19,"AnHui Jing Sai Technology Co., LTD.",BSE,CNY,None,None
5526,920982.BJ,920982,锦波生物,山西,医疗保健,山西锦波生物医药股份有限公司,JBSW,北交所,L,2023-07-20,N,杨霞,民营企业,2026-04-30 20:39:26,"Shanxi Jinbo Bio-Pharmaceutical Co., Ltd.",BSE,CNY,None,2026-04-30
5527,920985.BJ,920985,海泰新能,河北,电气设备,唐山海泰新能科技股份有限公司,HTXN,北交所,L,2022-08-08,N,王永,民营企业,2026-04-29 20:23:20,"Tangshan Haitai New Energy Technology Co., Ltd.",BSE,CNY,None,2026-04-29


In [7]:
df = db.qry2df("select * from jhd.trading_dates td")
df.columns = lookUpColName(df)

In [8]:
df

,唯一ID,交易日期,是否已经导入日交易数据
0,1,2005-01-04,None
1,2,2005-01-05,None
2,3,2005-01-06,None
3,4,2005-01-07,None
4,5,2005-01-10,None
...,...,...,...
5203,5208,2026-06-08,None
5204,5209,2026-06-09,None
5205,5210,2026-06-10,None
5206,5211,2026-06-11,None
